# colab_08d — Pivot: scanorama integration

## Motivation

Three Harmony attempts on the balanced 100k+100k inputs all failed the §11 RG check:

| Run | HVGs | theta | Iterations | Mean shift | >95%-pure | Dominant RG cluster | Fetal RG enrichment |
|---|---|---|---|---|---|---|---|
| colab_08  | 2,000 union     | 2 | 2 | 0.626 | ca. 41% | 0  (98.7% **2020**) | 6.66× |
| colab_08b | 751 intersect   | 2 | 1 | 0.127 | ca. 61% | 5  (87.2% **2021**) | 6.69× |
| colab_08c | 2,000 union     | 4 | 5 | 0.420 | ca. 41% | 13 (99.5% **2021**) | 4.53× |

Same failure mode rotated three ways: Harmony always produces one high-purity RG cluster, but which side dominates flips between runs. Pure-cluster fraction never drops. Conclusion from Session 23: **Harmony cannot bridge this cross-protocol gap regardless of theta or HVG choice**.

scVI was the planned escalation, but it requires raw integer counts. colab_07b confirmed that the GEO-archived Bhaduri 2020 expression matrix is library-size-normalized (`cellranger aggr --normalize=mapped`), not raw — so scVI is off the table without re-running cellranger from SRA fastqs (multi-day compute).

## This notebook: scanorama on the existing 100k files

[Hie et al. 2019 (Nat Biotech)](https://www.nature.com/articles/s41587-019-0113-3). Mutual-nearest-neighbor panoramic alignment. Works on log-normalized data — exactly what we have. No GPU, no raw counts. Runs in ca. 10 min on Colab standard RAM.

Why it might succeed where Harmony failed: scanorama matches per-cell anchors via MNN in expression space and aligns whole panoramas to those anchors, rather than minimizing a per-batch diversity penalty in PCA space. When the two batches differ by a discrete protocol effect (organoid vs primary tissue) on top of shared cell-type biology, MNN anchors can identify the genuinely shared subpopulations and align to them, instead of trying to evenly mix every cluster the way Harmony's objective does. If scanorama also pins one RG cluster to a single dataset, the asymmetry is biological (or technical in a way no integrator can fix) and we proceed to colab_10 annotation on colab_08's output, accepting the limitation.

Single-method change. Same 2,000 union HVGs (`flavor='seurat'`, `batch_key='dataset'`) as colab_08 to keep the comparison apples-to-apples — only the integration step differs.

## Outputs

- `integrated_100k_scanorama.h5ad` on Drive (separate from colab_08 / 08b / 08c outputs).
- §10 re-runs the cluster-0 fetal-RG diagnostic with the same logic as colab_08c §11. **Pass criterion**: dominant RG cluster has substantial mixing (neither >95% organoid nor >95% fetal); fetal RG enrichment ≳ 5×; cells in >95%-pure clusters drops below colab_08's 41%.
- **Fail routing**: if scanorama also fails, accept the asymmetry as biological/structural, proceed to colab_10 annotation on `integrated_100k_harmony.h5ad` (colab_08's output) and scope cross-dataset RG trajectory analysis to within-organoid only.

## 0. Setup

### 0a — Install dependencies, mount Drive, import packages

Installs `scanpy`, `leidenalg`, and `scanorama` (none are pre-installed on standard Colab images). scanorama pulls `annoy` (kNN) and `fbpca` (truncated SVD) as dependencies. Mounts Google Drive (project lives at `MyDrive/brain-organoid-trajectories`), imports the scanpy stack, and defines `PATHS` for both inputs and the integration output.

In [ ]:
!pip install -q scanpy leidenalg scanorama

from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import scanorama
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(6, 5))

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'bhaduri_2020_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'),
    'bhaduri_2021_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad'),
    'integrated_out':    os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_100k_scanorama.h5ad'),
}
os.makedirs(os.path.dirname(PATHS['integrated_out']), exist_ok=True)
print('scanpy', sc.__version__, '| anndata', ad.__version__, '| scanorama', scanorama.__version__ if hasattr(scanorama, '__version__') else 'n/a')

## 1. Load both 100k subsamples

### 1a — Read both h5ads from Drive

Loads each AnnData and prints shape, raw status, and obs columns. The two datasets store counts differently — we unify in section 2.

In [ ]:
adata_2020 = sc.read_h5ad(PATHS['bhaduri_2020_100k'])
adata_2021 = sc.read_h5ad(PATHS['bhaduri_2021_100k'])

print(f'Bhaduri 2020: {adata_2020.shape[0]:,} cells x {adata_2020.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2020.raw is not None else "None"})')
print(f'Bhaduri 2021: {adata_2021.shape[0]:,} cells x {adata_2021.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2021.raw is not None else "None"})')
print()
print('Bhaduri 2020 obs cols:', list(adata_2020.obs.columns))
print('Bhaduri 2021 obs cols:', list(adata_2021.obs.columns))

## 2. Bring both `.X` matrices to the same processing stage (log-normalized)

The two subsamples store data asymmetrically:
- **Bhaduri 2020**: `.raw` is a standard scanpy snapshot taken *after* normalize_total + log1p in colab_01. `.raw.X` is log(CP10K+1) data — **not raw counts** (verified in colab_08 cell 2b on 2026-04-25).
- **Bhaduri 2021**: `raw=False` from colab_07 — `.X` holds genuine raw integer counts.

Goal: bring both to log(CP10K+1) on the full gene space before concat. 2020 only needs `.X = .raw.X`; 2021 needs normalize_total + log1p applied. scanorama specifically expects log-normalized inputs (it is sensitive to scale, unlike methods that scale internally), so this stage matters more here than for Harmony.

### 2a — For Bhaduri 2020: copy `.raw.X` (log-normalized full-gene matrix) into `.X`

Replaces 2020's scaled-HVG `.X` (from colab_02) with the log-normalized full-gene snapshot from `.raw`. After this cell 2020 is in log(CP10K+1) on 16,774 genes.

In [ ]:
if adata_2020.raw is None:
    raise ValueError('Bhaduri 2020 .raw is unexpectedly None — re-check colab_07 output')
adata_2020.X = adata_2020.raw.X.copy()

if adata_2021.raw is not None:
    print('WARN: Bhaduri 2021 .raw unexpectedly set — still using .X (assumed counts)')

for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    if not sp.issparse(a.X):
        a.X = sp.csr_matrix(a.X)
    elif a.X.format != 'csr':
        a.X = a.X.tocsr()
    print(f'Bhaduri {name}: dtype={a.X.dtype}, format={a.X.format}, nnz={a.X.nnz:,}')

### 2b — Inspect `.X` ranges to confirm processing stage

Sanity check: 2020's `.X` should look like log-normalized data (float, max ca. 6–8, non-integer). 2021's `.X` should still be raw integer counts (max in tens or hundreds, integer-like).

In [ ]:
for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    n = min(1000, a.X.nnz)
    sample = a.X.data[:n]
    is_int_like = np.allclose(sample, sample.astype(int))
    print(f'Bhaduri {name}: min={sample.min()}, max={sample.max()}, '
          f'dtype={sample.dtype}, integer-like={is_int_like}')

### 2c — Normalize + log-transform Bhaduri 2021 to match 2020

`sc.pp.normalize_total(target_sum=1e4)` + `sc.pp.log1p`. Brings 2021 from raw integer counts into log(CP10K+1) on the full gene space, the same stage 2020 is at after 2a.

In [ ]:
sc.pp.normalize_total(adata_2021, target_sum=1e4)
sc.pp.log1p(adata_2021)
print(f'2021 .X after norm+log: dtype={adata_2021.X.dtype}, '
      f'min={adata_2021.X.min():.3f}, max={adata_2021.X.max():.2f}')

## 3. Shared gene space

### 3a — Intersect `var_names` across datasets

Integration requires identical gene panels. Expected from Session 20 sanity check: ca. 16,768 shared genes — the 6 Bhaduri-2020-only genes are `.1`-suffixed Cell Ranger duplicates and biologically irrelevant. Subset both datasets to the shared genes in sorted order.

In [ ]:
shared_genes = adata_2020.var_names.intersection(adata_2021.var_names).sort_values()

print(f'Bhaduri 2020 genes: {adata_2020.shape[1]:,}')
print(f'Bhaduri 2021 genes: {adata_2021.shape[1]:,}')
print(f'Shared:             {len(shared_genes):,}')
print(f'Lost from 2020:     {adata_2020.shape[1] - len(shared_genes):,}')
print(f'Lost from 2021:     {adata_2021.shape[1] - len(shared_genes):,}')

adata_2020 = adata_2020[:, shared_genes].copy()
adata_2021 = adata_2021[:, shared_genes].copy()

print()
print(f'After subset — 2020: {adata_2020.shape}')
print(f'After subset — 2021: {adata_2021.shape}')

## 4. Concatenate with dataset label

### 4a — Prefix barcodes and concat

Prefix obs names with `b2020_` / `b2021_` so combined barcodes are globally unique. `ad.concat(..., label='dataset')` adds an obs column named `dataset` with values `bhaduri_2020` / `bhaduri_2021` — this becomes scanorama's batch key when we split again in §6.

We concat first (rather than running scanorama directly on the per-dataset objects) so that HVG selection in §5 can use `batch_key='dataset'` to rank HVGs per dataset and merge — same logic as colab_08. We then split back into a per-dataset list for scanorama in 6a.

Dataset-specific obs columns (`protocol`, `age_week`, `cell_type_coarse`, `age_gw`, `donor`, `area_ucsc`) are not in both objects, so they will be `NaN`-filled. They are preserved through concat so the §10 RG diagnostic can use `cell_type_coarse` for fetal cells.

In [ ]:
adata_2020.obs_names = 'b2020_' + adata_2020.obs_names
adata_2021.obs_names = 'b2021_' + adata_2021.obs_names
print('First 3 barcodes 2020:', adata_2020.obs_names[:3].tolist())
print('First 3 barcodes 2021:', adata_2021.obs_names[:3].tolist())

adata = ad.concat(
    {'bhaduri_2020': adata_2020, 'bhaduri_2021': adata_2021},
    axis=0,
    join='outer',
    label='dataset',
    merge='same',
    index_unique=None,
)
adata.obs['dataset'] = adata.obs['dataset'].astype('category')

del adata_2020, adata_2021

print()
print(f'Combined: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')
print(adata.obs['dataset'].value_counts())

## 5. Select highly variable genes

Both datasets are already log(CP10K+1) (after 2a + 2c), so no normalization needed here. Just HVG selection per batch — same call as colab_08 / 08c so the input feature space is identical across all four integration runs.

### 5a — Confirm combined `.X` is log-normalized

Quick sanity check before HVG selection. Combined `.X` should be float, max ca. 6–8.

In [ ]:
print(f'Combined .X: dtype={adata.X.dtype}, '
      f'min={adata.X.min():.3f}, max={adata.X.max():.2f}, '
      f'mean={adata.X.mean():.3f}')

### 5b — Select 2,000 HVGs (seurat flavor, per-batch)

`flavor='seurat'` works on log-normalized `.X`. `batch_key='dataset'` ranks HVGs per dataset and merges. Save `.raw = adata` (full log-normalized 16,768-gene matrix) before subsetting `.X` to HVGs so the §10 RG-marker score and §8c marker plots can read from `.raw`.

Expected: 2,000 HVGs, ca. 750 HV in both batches and ca. 1,250 HV in one only — same distribution as colab_08.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor='seurat',
    batch_key='dataset',
)
adata.raw = adata
adata = adata[:, adata.var['highly_variable']].copy()
print(f'HVGs selected: {adata.shape[1]:,} genes')
print(f'Highly variable in N batches: '
      f'{adata.var["highly_variable_nbatches"].value_counts().sort_index().to_dict()}')

## 6. scanorama integration

scanorama operates differently from Harmony:
- **Input**: a list of per-batch AnnData objects with the same gene panel (HVGs).
- **Algorithm**: finds mutual-nearest-neighbor (MNN) anchors between batches in expression space, computes a panorama alignment that aligns each batch to the others through those anchors. Internally uses truncated SVD (`fbpca`) for efficiency and `annoy` for kNN search.
- **Output**: `obsm['X_scanorama']` on each input AnnData — a low-dim integrated embedding that replaces PCA for downstream neighbors / UMAP / Leiden.
- **No PCA pre-step needed**: scanorama computes its own SVD internally. We skip the explicit `sc.pp.scale` + `sc.tl.pca` that colab_08 ran before Harmony.
- **No theta-equivalent**: alignment strength is determined by the MNN k (default 20) and a similarity threshold (`sigma=15`, `alpha=0.10`) — we use scanorama defaults, since the failure mode in Harmony was unrelated to a tunable strength parameter.

We set `dimred=30` to match the prior PCA convention (colab_08 used 30 PCs). This keeps the downstream neighbor graph in the same dimensionality as the Harmony runs for fair comparison.

### 6a — Split combined adata back into per-dataset list

scanorama needs a list of separate AnnData objects, not a single concatenated one. Slice by `dataset` to recover the per-batch objects, ordered consistently. Each sub-object inherits the shared HVG gene panel from §5.

In [ ]:
batch_order = ['bhaduri_2020', 'bhaduri_2021']
adatas_split = [adata[adata.obs['dataset'] == b].copy() for b in batch_order]

for b, a in zip(batch_order, adatas_split):
    print(f'{b}: {a.shape[0]:,} cells x {a.shape[1]:,} genes  '
          f'(X dtype={a.X.dtype}, format={a.X.format if sp.issparse(a.X) else "dense"})')

### 6b — Run scanorama.integrate_scanpy

Mutates each AnnData in `adatas_split` in place, adding `obsm['X_scanorama']` of shape `(n_cells, dimred)`. `dimred=30` matches our PCA convention. Time the call — expected ca. 5–15 min on Colab standard RAM at 200k cells × 2,000 genes × 2 batches.

If this OOMs on standard RAM, restart with high-RAM (25 GB) — scanorama densifies HVG matrices internally, peak memory is ca. 2 × cells × HVGs × 4 bytes = ca. 3.2 GB plus working overhead, generally fits in 12 GB but can be tight if Colab has other notebooks loaded.

In [ ]:
t0 = time.time()
scanorama.integrate_scanpy(adatas_split, dimred=30)
elapsed = time.time() - t0

print(f'scanorama.integrate_scanpy: {elapsed:.1f}s ({elapsed/60:.1f} min)')
for b, a in zip(batch_order, adatas_split):
    print(f'  {b}: X_scanorama shape {a.obsm["X_scanorama"].shape}')

### 6c — Stitch X_scanorama back into combined adata

The per-dataset embeddings need to be reassembled into a single `(n_cells, 30)` matrix on the combined adata, with rows in the same order as `adata.obs_names`. Use `get_indexer` on barcodes to map each split row back to its row in the combined object.

After this cell `adata.obsm['X_scanorama']` is the integrated embedding for downstream neighbors / UMAP / Leiden.

In [ ]:
X_scan = np.zeros((adata.n_obs, 30), dtype=np.float32)
for sub in adatas_split:
    idx = adata.obs_names.get_indexer(sub.obs_names)
    if (idx < 0).any():
        raise RuntimeError(f'{(idx<0).sum()} barcodes from split not found in combined adata')
    X_scan[idx] = sub.obsm['X_scanorama']
adata.obsm['X_scanorama'] = X_scan

del adatas_split
print(f'X_scanorama on combined adata: shape {adata.obsm["X_scanorama"].shape}, '
      f'dtype {adata.obsm["X_scanorama"].dtype}')

### 6d — Embedding sanity check

Verify the embedding has nonzero variance across all 30 dims and reasonable per-cell magnitudes. A degenerate run (e.g., all zeros for one batch, or one dimension dominating) shows up here before we waste time on neighbors / UMAP. There is no Harmony-style 'mean per-cell shift' diagnostic for scanorama because the input space (HVG expression) and output space (low-dim panorama embedding) are different — the relevant verdict is the visual mixing in 8a and the §10 RG check.

In [ ]:
X = adata.obsm['X_scanorama']
print(f'Per-dim std (first 5):  {np.round(X.std(axis=0)[:5], 4)}')
print(f'Per-dim std (last 5):   {np.round(X.std(axis=0)[-5:], 4)}')
print(f'Min std across dims:    {X.std(axis=0).min():.4f}')
print(f'Per-cell norm — median: {np.linalg.norm(X, axis=1).mean():.3f}, '
      f'std: {np.linalg.norm(X, axis=1).std():.3f}')
print()
for b in batch_order:
    mask = (adata.obs['dataset'] == b).values
    sub = X[mask]
    print(f'{b}: mean per-dim |val|={np.abs(sub).mean():.4f}, std={sub.std():.4f}')

## 7. Neighbor graph, UMAP, Leiden

### 7a — Compute neighbor graph on scanorama embedding

`use_rep='X_scanorama'` so neighbors reflect batch-corrected distances. `n_neighbors=15` is scanpy default — same as colab_08 / 08b / 08c.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_scanorama', random_state=42)
print('Neighbor graph computed on X_scanorama.')

### 7b — UMAP

2D embedding for inspection. Random state fixed for reproducibility.

In [ ]:
sc.tl.umap(adata, random_state=42)
print(f'X_umap shape: {adata.obsm["X_umap"].shape}')

### 7c — Leiden clustering at resolution 0.5

Matches colab_08 / 08b / 08c. Expected count: ca. 15–25 clusters. The exact number is not the key signal — the §10 RG composition check is.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, random_state=42)
n_clusters = adata.obs['leiden'].nunique()
print(f'Leiden clusters: {n_clusters}')
print(adata.obs['leiden'].value_counts().sort_index())

## 8. Visual diagnostics

### 8a — UMAP colored by `dataset` (mixing check)

If scanorama worked, the two colors should intermix within each cell-type region. Stripes / dataset-segregated regions = scanorama failed or under-corrected. This is the visual analogue of Harmony's per-cell shift diagnostic.

In [ ]:
sc.pl.umap(adata, color='dataset', frameon=False, save='_dataset.png', show=True)

### 8b — UMAP colored by Leiden cluster

Cluster geography. Cell-type assignment happens in colab_10 (next notebook) if §10 passes.

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False, save='_leiden.png', show=True)

### 8c — Lineage marker genes

Sanity-check that markers land in expected regions:
- **SOX2**, **PAX6** — radial glia
- **EOMES** — intermediate progenitors
- **TBR1**, **NEUROD2** — excitatory neurons
- **GAD1**, **GAD2** — GABAergic interneurons
- **GFAP** — astrocytes
- **MKI67** — cycling cells

Scattered/diffuse markers → over-integration. `use_raw=True` reads from the full 16,768-gene `.raw` matrix, not the 2,000-HVG `.X`.

**Two outputs per marker run:**
1. A combined 3×3 grid panel (`_markers.png`) — good for at-a-glance comparison.
2. Individual full-size plots per gene (`_marker_<GENE>.png`) — needed for accurate per-gene spatial reads.

In [ ]:
markers = ['SOX2', 'PAX6', 'EOMES', 'TBR1', 'NEUROD2', 'GAD1', 'GAD2', 'GFAP', 'MKI67']
present = [g for g in markers if g in adata.raw.var_names]
missing = [g for g in markers if g not in adata.raw.var_names]
if missing:
    print(f'Markers missing from .raw: {missing}')

sc.pl.umap(adata, color=present, use_raw=True, ncols=3, frameon=False,
           save='_markers.png', show=True)

for gene in present:
    sc.pl.umap(adata, color=gene, use_raw=True, frameon=False, size=8,
               save=f'_marker_{gene}.png', show=True)

### 8d — Cluster × dataset composition

Percentage of each dataset within each Leiden cluster. Values near 50/50 mean scanorama mixed that cluster well; clusters > 95% one dataset are candidates for batch artifacts. The headline metric for §10 is the count of cells in >95%-pure clusters — colab_08 had ca. 41%, colab_08b had ca. 61%, colab_08c had ca. 41%.

In [ ]:
comp = pd.crosstab(adata.obs['leiden'], adata.obs['dataset'], normalize='index') * 100
comp = comp.round(1)
comp['total_cells'] = adata.obs['leiden'].value_counts().sort_index().values
print(comp)

## 9. Save integrated object

### 9a — Write `integrated_100k_scanorama.h5ad` to Drive

Stored alongside the three Harmony outputs (`integrated_100k_harmony.h5ad`, `_intersectHVG.h5ad`, `_theta4.h5ad`) for direct comparison. Predicted size: ca. 6–7 GB (same shape as colab_08; only the embedding values differ — `obsm['X_scanorama']` replaces `obsm['X_pca_harmony']`).

In [ ]:
adata.write_h5ad(PATHS['integrated_out'])
size_gb = os.path.getsize(PATHS['integrated_out']) / 1e9
print(f'Saved: {PATHS["integrated_out"]}')
print(f'Size:  {size_gb:.2f} GB')
print(f'Final shape: {adata.shape}')

## 10. RG remediation check

Same logic as colab_08c §11 — find the dominant RG cluster, measure its dataset composition and the fetal-side cell-type-coarse enrichment, and decide whether scanorama escapes the three-rotation Harmony failure.

### 10a — Find the dominant RG cluster

Identify which cluster has the largest mean PAX6 + SOX2 + MKI67 + VIM + NES + FABP7 score, taken on `.raw` (full-gene log-normalized). This cluster is what we then compositionally probe in 10b.

In [ ]:
rg_markers_check = ['PAX6', 'SOX2', 'MKI67', 'VIM', 'NES', 'FABP7']
rg_idx = [adata.raw.var_names.get_loc(g) for g in rg_markers_check if g in adata.raw.var_names]
rg_score_per_cell = adata.raw.X[:, rg_idx]
if hasattr(rg_score_per_cell, 'toarray'):
    rg_score_per_cell = rg_score_per_cell.toarray()
rg_score_per_cell = rg_score_per_cell.mean(axis=1)

df = pd.DataFrame({
    'leiden':   adata.obs['leiden'].values,
    'rg_score': rg_score_per_cell,
    'dataset':  adata.obs['dataset'].values,
})
cluster_rg = df.groupby('leiden')['rg_score'].mean().sort_values(ascending=False)
print('Top 5 clusters by mean RG-marker score:')
print(cluster_rg.head().round(3).to_string())

rg_cluster = cluster_rg.index[0]
print(f'\nDominant RG cluster: {rg_cluster}')
print(df[df['leiden']==rg_cluster]['dataset'].value_counts().to_string())

### 10b — Fetal cell_type_coarse enrichment in the dominant RG cluster

Same diagnostic as colab_08c §11b. **Pass criteria** for scanorama remediation:
- Fetal cells in dominant RG cluster substantially higher than colab_08's 298 (target: at least a few thousand of the ca. 11,200 fetal RG should land here).
- Fraction of all 2020 cells in this cluster ≈ fraction of all 2021 RG in this cluster (similar absolute mixing on both sides).
- RG enrichment ratio in the fetal contingent stays high (≳ 5×) — confirms it's still an RG cluster, not a degenerated mixed cluster.
- Dominant RG cluster is **not** >95% organoid-pure and **not** >95% fetal-pure (the consistent Harmony failure mode).

In [ ]:
mask_rg      = (adata.obs['leiden'] == rg_cluster).values
mask_2021    = (adata.obs['dataset'] == 'bhaduri_2021').values
mask_2020    = (adata.obs['dataset'] == 'bhaduri_2020').values
mask_in_2021 = mask_rg & mask_2021
mask_in_2020 = mask_rg & mask_2020

print(f'Cluster {rg_cluster} composition:')
print(f'  bhaduri_2020: {mask_in_2020.sum():,} cells '
      f'(= {mask_in_2020.sum() / mask_2020.sum() * 100:.1f}% of all 2020)')
print(f'  bhaduri_2021: {mask_in_2021.sum():,} cells '
      f'(= {mask_in_2021.sum() / mask_2021.sum() * 100:.1f}% of all 2021)')
print(f'  colab_08:  cluster 0  had 23,439 / 298    (98.7% organoid)')
print(f'  colab_08b: cluster 5  had 1,969 / 13,401  (87.2% fetal)')
print(f'  colab_08c: cluster 13 had ~120 / ~24,000  (99.5% fetal)')
print()

baseline   = adata.obs.loc[mask_2021,    'cell_type_coarse'].value_counts(normalize=True)
in_cluster = adata.obs.loc[mask_in_2021, 'cell_type_coarse'].value_counts(normalize=True)
enrichment = (in_cluster / baseline).sort_values(ascending=False)

print(f'Bhaduri 2021 baseline cell_type_coarse (n = {mask_2021.sum():,}):')
print(baseline.round(3).to_string())
print()
print(f'Cluster {rg_cluster} — Bhaduri 2021 cells only (n = {mask_in_2021.sum():,}):')
print(in_cluster.round(3).to_string())
print()
print(f'Enrichment ratio (>1 = enriched):')
print(enrichment.round(2).to_string())

### 10c — Verdict and routing

Decision rule:
- **Pass** (dominant RG cluster mixed AND fetal RG enrichment ≳ 5× AND pure-cluster fraction drops below 41%) → scanorama bridged the gap. Save this output as the integration of record. Proceed to colab_10 (full annotation) on `integrated_100k_scanorama.h5ad`.
- **Partial** (mixing improved but still skewed, or pure-cluster fraction drops only modestly) → scanorama is the best of the four; proceed to colab_10 on this output but document the limitation in NOTES.
- **Fail** (dominant RG cluster still >95% one-side, pure-cluster fraction still ca. 41%+) → all four integration approaches have hit the same wall. Accept the asymmetry as biological/structural, fall back to colab_08's output for colab_10 annotation, and scope the cross-dataset RG trajectory analysis to within-organoid only.

In [ ]:
n_pure_2020 = sum(1 for cl in adata.obs['leiden'].unique()
                  if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2020').mean() > 0.95)
n_pure_2021 = sum(1 for cl in adata.obs['leiden'].unique()
                  if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2021').mean() > 0.95)
total_pure_cells = sum(
    (adata.obs['leiden']==cl).sum()
    for cl in adata.obs['leiden'].unique()
    if (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2020').mean() > 0.95
    or (adata.obs.loc[adata.obs['leiden']==cl, 'dataset']=='bhaduri_2021').mean() > 0.95
)

print('=' * 70)
print('REMEDIATION SUMMARY — scanorama')
print('=' * 70)
print(f'Method:                           scanorama (MNN panorama, dimred=30)')
print(f'HVG count:                        {adata.shape[1]:,}  (vs colab_08\'s 2,000 union)')
print(f'Leiden clusters:                  {adata.obs["leiden"].nunique()}  (vs colab_08\'s 21)')
print(f'>95%-pure clusters: 2020={n_pure_2020}, 2021={n_pure_2021}')
print(f'Cells in >95%-pure clusters:      {total_pure_cells:,} '
      f'({total_pure_cells/len(adata)*100:.0f}%)  (vs colab_08 41% / 08b 61% / 08c 41%)')
print(f'Dominant RG cluster:              {rg_cluster}')
print(f'  composition (2020 / 2021):      {mask_in_2020.sum():,} / {mask_in_2021.sum():,}')
print(f'  fetal RG enrichment:            {enrichment.get("RG", 0):.2f}x  '
      f'(vs colab_08 6.66x / 08b 6.69x / 08c 4.53x)')
print('=' * 70)